# Scientific Computing Code Sample: Data-Driven Materials Degradation

**Purpose.** This notebook demonstrates how I structure, train, validate, and physically interrogate a computational model for a materials-degradation problem.

The underlying application is corrosion inhibition of API 5L X52 carbon steel in CO₂-loaded monoethanolamine (MEA). Although the physical system differs from fracture/fatigue mechanics, the notebook is intended to demonstrate transferable computational practice:

- explicit problem definition and data interfaces;
- reproducible preprocessing and train/test separation;
- a custom PyTorch neural-network implementation;
- regularised optimisation and architecture selection;
- modular evaluation functions;
- physical-consistency checks;
- model interpretability;
- applicability-domain assessment; and
- attention to computational cost and reproducibility.

> **Confidentiality:** the experimental dataset is associated with ongoing work currently under peer review and is not distributed with this code sample. The notebook is executable when an authorised local copy is placed in the documented data location.

## 1. Computational setup

A deterministic seed is used for reproducibility. The code detects CPU/GPU availability automatically so the same workflow can run on a workstation or a CUDA-enabled environment.

In [ ]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn

SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")

## 2. Problem definition and data interface

The model maps five physically meaningful operating variables to inhibition efficiency:

\[
\mathbf{x} = [C_{inh}, C_{MEA}, T, t, pH]
\quad \longrightarrow \quad IE
\]

where inhibitor concentration, MEA concentration, temperature, immersion time, and pH define the input state.

The loader below performs explicit schema checking and numeric validation before modelling begins.

In [ ]:
DATA_PATH = Path("../data/ANN_dataset_cleaned.xlsx")

REQUIRED_COLUMNS = {
    "Inhibitor_Concentration (mL)": "Inhibitor_Concentration",
    "MEA_Concentration (wt%)": "MEA_Concentration",
    "Temperature (K)": "Temperature",
    "Immersion_Time (h)": "Immersion_Time",
    "pH": "pH",
    "IE (%)": "IE",
}

INPUT_COLS = [
    "Inhibitor_Concentration",
    "MEA_Concentration",
    "Temperature",
    "Immersion_Time",
    "pH",
]
TARGET_COL = "IE"

def load_research_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            "Experimental data are intentionally excluded from this application sample. "
            "Place an authorised local copy at ../data/ANN_dataset_cleaned.xlsx."
        )

    raw = pd.read_excel(path)
    raw.columns = raw.columns.str.strip()

    missing = [c for c in REQUIRED_COLUMNS if c not in raw.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = raw[list(REQUIRED_COLUMNS)].rename(columns=REQUIRED_COLUMNS).copy()

    for column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    if df.isna().any().any():
        bad = df.isna().sum()
        raise ValueError(
            "Missing/non-numeric values detected:\n"
            + str(bad[bad > 0])
        )

    return df

df = load_research_data(DATA_PATH)
print(f"Validated dataset: {df.shape[0]} observations, {df.shape[1]} variables")

## 3. Preprocessing and reproducible data split

Inputs and target are scaled to \([-1,1]\), matching the hidden-layer `tanh` activation. The train/test split is fixed by seed so architecture comparisons are made on the same observations.

The scalers are fit on the **training set only** to avoid information leakage.

In [ ]:
def prepare_data(df: pd.DataFrame, test_size: float = 0.20, seed: int = SEED):
    X = df[INPUT_COLS].to_numpy(dtype=np.float32)
    y = df[[TARGET_COL]].to_numpy(dtype=np.float32)

    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X, y, test_size=test_size, random_state=seed
    )

    scaler_X = MinMaxScaler(feature_range=(-1, 1))
    scaler_y = MinMaxScaler(feature_range=(-1, 1))

    X_train = scaler_X.fit_transform(X_train_raw)
    X_test = scaler_X.transform(X_test_raw)
    y_train = scaler_y.fit_transform(y_train_raw)
    y_test = scaler_y.transform(y_test_raw)

    tensors = {
        "X_train": torch.tensor(X_train, dtype=torch.float32, device=DEVICE),
        "X_test": torch.tensor(X_test, dtype=torch.float32, device=DEVICE),
        "y_train": torch.tensor(y_train, dtype=torch.float32, device=DEVICE),
        "y_test": torch.tensor(y_test, dtype=torch.float32, device=DEVICE),
    }

    arrays = {
        "X_train_raw": X_train_raw,
        "X_test_raw": X_test_raw,
        "y_train_raw": y_train_raw,
        "y_test_raw": y_test_raw,
        "X_train_scaled": X_train,
        "X_test_scaled": X_test,
    }

    return tensors, arrays, scaler_X, scaler_y

T, A, scaler_X, scaler_y = prepare_data(df)
print(f"Training observations: {len(A['X_train_raw'])}")
print(f"Testing observations : {len(A['X_test_raw'])}")

## 4. Model definition

The network is intentionally compact: one hidden layer with `tanh` activation and one linear output. This makes the fitted model inspectable and permits direct extraction of its weights and biases.

The regularised objective is

\[
F = eta E_D + lpha E_W
\]

where \(E_D\) is the data misfit and \(E_W\) is the squared-weight penalty. During training, \(lpha\) and \(eta\) are updated periodically using an effective-parameter approximation.

In [ ]:
class RegularisedMLP(nn.Module):
    def __init__(self, n_inputs: int, n_hidden: int, n_outputs: int = 1):
        super().__init__()
        self.hidden = nn.Linear(n_inputs, n_hidden)
        self.output = nn.Linear(n_hidden, n_outputs)
        nn.init.xavier_uniform_(self.hidden.weight)
        nn.init.xavier_uniform_(self.output.weight)

    def forward(self, x):
        return self.output(torch.tanh(self.hidden(x)))


def train_model(
    model: nn.Module,
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    epochs: int = 1000,
    learning_rate: float = 1e-2,
    update_interval: int = 10,
):
    model = model.to(DEVICE)
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)

    n_obs = X_train.shape[0]
    n_params = sum(p.numel() for p in model.parameters())

    alpha = torch.tensor(1e-3, dtype=torch.float32, device=DEVICE)
    beta = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)

    history = []

    for epoch in range(epochs):
        model.train()
        optimiser.zero_grad()

        prediction = model(X_train)
        data_error = torch.sum((prediction - y_train) ** 2)
        weight_error = sum(torch.sum(p ** 2) for p in model.parameters())

        objective = beta * data_error + alpha * weight_error
        objective.backward()
        optimiser.step()

        if epoch % update_interval == 0:
            with torch.no_grad():
                gamma = n_params * beta / (beta + alpha)
                alpha = gamma / (2.0 * weight_error + 1e-10)
                beta = (n_obs - gamma) / (2.0 * data_error + 1e-10)

                alpha = torch.clamp(alpha, 1e-6, 1e3)
                beta = torch.clamp(beta, 1e-6, 1e6)

        history.append((data_error / n_obs).item())

    return {
        "model": model,
        "history": history,
        "alpha": float(alpha.detach().cpu()),
        "beta": float(beta.detach().cpu()),
    }

## 5. Architecture search

The search is deliberately parameterised rather than hard-coded. For an application code sample, the defaults below are modest enough for inspection; the research study used a broader repeated search.

The criterion is mean held-out test MSE across repeated initialisations. This is a pragmatic model-selection procedure for a small experimental dataset.

In [ ]:
def predict_scaled(model: nn.Module, X: torch.Tensor) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        return model(X).detach().cpu().numpy()


def architecture_search(
    hidden_sizes=range(2, 11),
    trials_per_size=5,
    epochs=300,
):
    results = []

    for n_hidden in hidden_sizes:
        trial_mse = []

        for trial in range(trials_per_size):
            set_seed(trial)

            model = RegularisedMLP(
                n_inputs=len(INPUT_COLS),
                n_hidden=n_hidden,
            )

            fit = train_model(
                model,
                T["X_train"],
                T["y_train"],
                epochs=epochs,
            )

            pred = predict_scaled(fit["model"], T["X_test"])
            mse = mean_squared_error(
                T["y_test"].detach().cpu().numpy(),
                pred,
            )
            trial_mse.append(mse)

        results.append({
            "n_hidden": n_hidden,
            "mean_test_mse": float(np.mean(trial_mse)),
            "std_test_mse": float(np.std(trial_mse)),
            "best_trial": int(np.argmin(trial_mse)),
        })

    return pd.DataFrame(results).sort_values("mean_test_mse").reset_index(drop=True)


search_table = architecture_search()
best_architecture = int(search_table.loc[0, "n_hidden"])
best_seed = int(search_table.loc[0, "best_trial"])

print(search_table)
print(f"Selected hidden-layer width: {best_architecture}")

## 6. Final fit and convergence

The selected architecture is retrained using the best initialisation identified during architecture search. The training history provides a direct convergence diagnostic.

In [ ]:
set_seed(best_seed)

final_fit = train_model(
    RegularisedMLP(len(INPUT_COLS), best_architecture),
    T["X_train"],
    T["y_train"],
    epochs=1000,
)

final_model = final_fit["model"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(final_fit["history"])
ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean squared data error")
ax.set_title("Training convergence")
plt.tight_layout()
plt.show()

## 7. Quantitative verification

Evaluation is performed in the original engineering units after inverse transformation. Both training and held-out testing performance are reported so overfitting can be assessed directly.

In [ ]:
def evaluate_model(
    model: nn.Module,
    X_tensor: torch.Tensor,
    y_scaled: torch.Tensor,
    scaler_y: MinMaxScaler,
):
    pred_scaled = predict_scaled(model, X_tensor)
    true_scaled = y_scaled.detach().cpu().numpy()

    y_pred = scaler_y.inverse_transform(pred_scaled).ravel()
    y_true = scaler_y.inverse_transform(true_scaled).ravel()

    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "y_true": y_true,
        "y_pred": y_pred,
    }


train_metrics = evaluate_model(
    final_model, T["X_train"], T["y_train"], scaler_y
)
test_metrics = evaluate_model(
    final_model, T["X_test"], T["y_test"], scaler_y
)

summary = pd.DataFrame(
    {
        "Training": {
            "R2": train_metrics["R2"],
            "RMSE": train_metrics["RMSE"],
            "MAE": train_metrics["MAE"],
        },
        "Testing": {
            "R2": test_metrics["R2"],
            "RMSE": test_metrics["RMSE"],
            "MAE": test_metrics["MAE"],
        },
    }
).T

print(summary.round(4))

## 8. Physical-consistency checks

Statistical accuracy alone is insufficient for an engineering model. I therefore probe the fitted response while holding all but one variable fixed.

The function below supports one-at-a-time sweeps over any model input. This makes it straightforward to compare predicted trends against known physical expectations and to flag non-physical behaviour.

In [ ]:
def predict_physical(model, X_raw, scaler_X, scaler_y):
    X_scaled = scaler_X.transform(np.asarray(X_raw, dtype=np.float32))
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=DEVICE)
    y_scaled = predict_scaled(model, X_tensor)
    return scaler_y.inverse_transform(y_scaled).ravel()


def one_factor_sweep(
    feature: str,
    values: np.ndarray,
    reference_state: dict,
):
    if feature not in INPUT_COLS:
        raise ValueError(f"Unknown feature: {feature}")

    rows = []

    for value in values:
        state = reference_state.copy()
        state[feature] = value
        rows.append([state[name] for name in INPUT_COLS])

    predictions = predict_physical(
        final_model,
        np.asarray(rows),
        scaler_X,
        scaler_y,
    )

    return pd.DataFrame({feature: values, "Predicted_IE": predictions})


reference_state = {
    col: float(df[col].median())
    for col in INPUT_COLS
}

feature = "Temperature"
values = np.linspace(df[feature].min(), df[feature].max(), 50)
trend = one_factor_sweep(feature, values, reference_state)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(trend[feature], trend["Predicted_IE"])
ax.set_xlabel(feature)
ax.set_ylabel("Predicted inhibition efficiency")
ax.set_title("Example one-factor physical trend check")
plt.tight_layout()
plt.show()

## 9. Explicit model inspection

Because the network has a single hidden layer, its learned parameters can be extracted directly. This is useful for auditing the fitted function and reproducing it outside PyTorch.

For normalised input vector \(\mathbf{x}_n\),

\[
IE_n = \mathbf{w}_2^	op 	anh(\mathbf{W}_1 \mathbf{x}_n + \mathbf{b}_1) + b_2.
\]

In [ ]:
def extract_parameters(model: RegularisedMLP):
    return {
        "W1": model.hidden.weight.detach().cpu().numpy().copy(),
        "b1": model.hidden.bias.detach().cpu().numpy().copy(),
        "W2": model.output.weight.detach().cpu().numpy().copy(),
        "b2": model.output.bias.detach().cpu().numpy().copy(),
    }

params = extract_parameters(final_model)

for name, array in params.items():
    print(f"{name}: shape={array.shape}")

## 10. Applicability domain

A model should not be trusted indiscriminately outside the region represented by its training data. I use a Williams-style leverage/residual diagnostic to identify statistically unusual observations and define a simple domain-of-validity check.

This is especially important for small experimental datasets, where apparently strong global metrics can hide local extrapolation.

In [ ]:
def williams_diagnostics(
    model,
    df,
    scaler_X,
    scaler_y,
):
    X_raw = df[INPUT_COLS].to_numpy(dtype=np.float32)
    y_raw = df[[TARGET_COL]].to_numpy(dtype=np.float32)

    X_scaled = scaler_X.transform(X_raw)
    y_scaled = scaler_y.transform(y_raw).ravel()

    X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=DEVICE)
    pred_scaled = predict_scaled(model, X_tensor).ravel()

    residual = y_scaled - pred_scaled
    standardised_residual = (
        residual - residual.mean()
    ) / (residual.std(ddof=1) + 1e-10)

    # Add an intercept column before computing leverage.
    X_design = np.column_stack([np.ones(len(X_scaled)), X_scaled])
    H = X_design @ np.linalg.pinv(X_design.T @ X_design) @ X_design.T
    leverage = np.diag(H)

    n = X_design.shape[0]
    p = X_design.shape[1] - 1
    h_star = 3 * (p + 1) / n

    diagnostics = pd.DataFrame({
        "leverage": leverage,
        "standardised_residual": standardised_residual,
    })

    diagnostics["within_residual_limit"] = (
        diagnostics["standardised_residual"].abs() <= 3
    )
    diagnostics["within_leverage_limit"] = (
        diagnostics["leverage"] <= h_star
    )
    diagnostics["within_domain"] = (
        diagnostics["within_residual_limit"]
        & diagnostics["within_leverage_limit"]
    )

    return diagnostics, h_star


diagnostics, h_star = williams_diagnostics(
    final_model, df, scaler_X, scaler_y
)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(
    diagnostics["leverage"],
    diagnostics["standardised_residual"],
    s=35,
)
ax.axhline(3, linestyle="--")
ax.axhline(-3, linestyle="--")
ax.axvline(h_star, linestyle="--")
ax.set_xlabel("Leverage")
ax.set_ylabel("Standardised residual")
ax.set_title("Williams applicability-domain diagnostic")
plt.tight_layout()
plt.show()

print(
    f"Observations within defined domain: "
    f"{diagnostics['within_domain'].mean() * 100:.1f}%"
)

## 11. Optional interpretability hook

For the research study, SHAP was also used to quantify feature influence. I keep this secondary to the physical-response checks because, for engineering use, a feature-attribution score is most useful when it agrees with a physically defensible response.

A SHAP implementation can be added without altering the modelling architecture above.

## 12. Computational considerations

A few choices in this workflow are deliberate:

- **Reproducibility:** seeds are fixed for NumPy, Python, and PyTorch.
- **Leakage control:** preprocessing parameters are fitted only on the training partition.
- **Scaling:** \([-1,1]\) scaling is consistent with the `tanh` hidden activation.
- **Model complexity:** architecture width is treated as a hyperparameter rather than assumed.
- **Regularisation:** the optimisation penalises model weights while adapting the relative weight of data and parameter error.
- **Convergence:** the training error history is retained and inspected.
- **Device portability:** the same code runs on CPU or CUDA.
- **Validity:** held-out error and leverage/residual diagnostics are both used before interpreting predictions.
- **Physical reasoning:** one-factor sweeps are used to identify non-physical trends that conventional ML metrics may miss.
- **Computational cost:** architecture search is isolated in a reusable function so trial counts and epochs can be scaled according to available compute resources.

The present samples were developed primarily on workstation and cloud-based environments. I am extending this workflow toward traditional HPC/cluster execution for larger numerical problems.

## 13. Relevance to computational mechanics

This notebook comes from a corrosion/materials-degradation problem rather than fracture mechanics. I include it because it reflects the computational habits I would bring to a phase-field fracture/fatigue problem:

1. define the governing input/output problem explicitly;
2. separate model construction from evaluation;
3. build reusable numerical functions rather than rely on one-off notebook cells;
4. check convergence and generalisation;
5. interrogate results for physical consistency;
6. identify the region in which a model is trustworthy; and
7. make computational cost and reproducibility explicit.

My objective in transitioning to phase-field fracture/fatigue modelling is to apply this same discipline to PDE-based computational mechanics, while developing deeper expertise in variational fracture formulations, nonlinear solvers, parallel computing, and titanium-alloy fatigue.